<!-- CELL 00 — Notebook introduction -->

<div style="
    background:#fff9db;
    border:1px solid #eadf9a;
    border-left:5px solid #d6b656;
    padding:22px 26px;
    margin:8px 0 22px 0;
    border-radius:6px;
">

<h1 style="margin-top:0; margin-bottom:6px;">
Battery Dispatch and the Economic Value of Load Forecasts
</h1>

<p style="font-size:1.08em; margin-top:0;">
<b>From forecast accuracy to decision quality and realized economic value</b>
</p>

<hr style="border:none; border-top:1px solid #e2d690; margin:18px 0;">

<h3>Research question</h3>

<p>
Does improved statistical accuracy in day-ahead load forecasting translate
into improved economic battery dispatch, and under what price, battery-state,
and constraint conditions can the ranking of forecasts change?
</p>

<p>
A deliberately provocative formulation is:
</p>

<p style="font-size:1.10em; text-align:center;">
<b>Does a 13.6% MAE gap imply a 13.6% economic-value gap?</b>
</p>

<h3>Research structure</h3>

<p style="text-align:center; font-size:1.10em;">
<b>
forecast
&nbsp;&rarr;&nbsp;
uncertainty
&nbsp;&rarr;&nbsp;
dispatch decision
&nbsp;&rarr;&nbsp;
realized economic consequence
</b>
</p>

<p>
The objective is not to construct a forecasting or optimization
leaderboard. The experiment asks which forecast errors alter physical
decisions, when they have economic consequences, and why.
</p>

<h3>Notebook roadmap</h3>

<ol>
    <li><b>Research and temporal-integrity contract</b></li>
    <li><b>Core battery and tariff parameters</b></li>
    <li><b>Belgian day-ahead prices and information-integrity gate</b></li>
    <li><b>When does a load forecast have decision value?</b></li>
    <li><b>Minimal battery-dispatch LP</b></li>
    <li><b>Independent verification and falsification tests</b></li>
    <li><b>Forecast inputs and stylised site construction</b></li>
    <li><b>Economic evaluation and regret</b></li>
    <li><b>Dual-based attribution of forecast errors</b></li>
    <li><b>Untouched confirmation</b></li>
    <li><b>Dependence-aware inference and sensitivity analysis</b></li>
</ol>

<p style="margin-bottom:0;">
<b>Design objective:</b>
scientific defensibility, temporal integrity, attribution,
mathematical clarity and reproducibility — not the production
of an impressive-looking positive result.
</p>

</div>

<!-- CELL 01 — Research and temporal-integrity contract -->

<div style="
    background:#fff9db;
    border-left:4px solid #d6b656;
    padding:18px 22px;
    margin:12px 0 18px 0;
    border-radius:4px;
">

<h2 style="margin-top:0;">Research Contract</h2>

<p>
This project studies the <b>economic decision value of day-ahead load forecasts</b>,
rather than forecasting accuracy in isolation.
</p>

<p>
The primary estimand is the difference in realized economic performance produced
by alternative D−1 load forecasts when all other elements of the battery-dispatch
problem are held fixed.
</p>

<h3>Temporal-integrity principle</h3>

<p>
The operational decision origin is:
</p>

<p style="text-align:center; font-size:1.10em;">
<b>18:00 Europe/Brussels on D−1</b>
</p>

<p>
Only information available at that origin may determine the operational
dispatch for delivery day D.
</p>

<p style="text-align:center; font-size:1.08em;">
<b>
Forecast information determines dispatch.<br>
Realized outcomes determine ex-post economic value.
</b>
</p>

<p>
Realized future load, generation, prices, or other post-origin information
must never enter an operational optimization. Such information may enter
only ex-post evaluation or an explicitly labelled
<b>perfect-foresight benchmark</b>.
</p>

<h3>Version-1 experimental contract</h3>

<ul>
    <li>One Belgian civil delivery day is optimized at a time.</li>
    <li>The physical time grid is quarter-hourly.</li>
    <li>DST days retain their actual 92, 96, or 100 quarter-hours.</li>
    <li>Dispatch is planned at 18:00 D−1 and executed open-loop.</li>
    <li>The grid absorbs deviations between forecast and realized site load.</li>
    <li>Initial and terminal battery state of charge are equal.</li>
    <li>No monthly peak tariff is included in version 1.</li>
    <li>
        Import/export price asymmetry is the primary mechanism through
        which load forecasts acquire economic decision value.
    </li>
    <li>
        Grid import/export capacity constraints are reserved for
        sensitivity analysis.
    </li>
    <li>
        Perfect foresight is an unattainable benchmark, never an
        operational policy.
    </li>
</ul>

<h3>Validation principle</h3>

<p>
Important conclusions must be independently checked wherever practical.
Analytical results are accompanied by numerical falsification tests,
and optimization results are not accepted solely because a solver
reports success.
</p>

<h3>Development discipline</h3>

<ul>
    <li>Research logic belongs in tested package code, not only in the notebook.</li>
    <li>The notebook orchestrates experiments and records the research argument.</li>
    <li>Information-boundary assumptions must fail loudly when violated.</li>
    <li>Temporary diagnostic cells are deleted before commit.</li>
    <li>Long-running experiments report progress while executing.</li>
    <li>
        Primary specifications are frozen before untouched confirmation
        results are opened.
    </li>
</ul>

</div>

In [1]:
# CELL 02 — Core battery and tariff parameters

from dataclasses import dataclass


@dataclass(frozen=True)
class BatterySpec:
    """
    Physical battery specification.

    Units
    -----
    energy_capacity_kwh  : kWh
    max_charge_kw        : kW
    max_discharge_kw     : kW
    charge_efficiency    : dimensionless
    discharge_efficiency : dimensionless
    initial_soc_fraction : fraction of usable energy capacity
    """

    energy_capacity_kwh: float = 100.0
    max_charge_kw: float = 50.0
    max_discharge_kw: float = 50.0
    charge_efficiency: float = 0.95
    discharge_efficiency: float = 0.95
    initial_soc_fraction: float = 0.50

    def __post_init__(self):
        if self.energy_capacity_kwh <= 0:
            raise ValueError("energy_capacity_kwh must be strictly positive.")

        if self.max_charge_kw <= 0:
            raise ValueError("max_charge_kw must be strictly positive.")

        if self.max_discharge_kw <= 0:
            raise ValueError("max_discharge_kw must be strictly positive.")

        if not 0 < self.charge_efficiency <= 1:
            raise ValueError("charge_efficiency must lie in (0, 1].")

        if not 0 < self.discharge_efficiency <= 1:
            raise ValueError("discharge_efficiency must lie in (0, 1].")

        if not 0 <= self.initial_soc_fraction <= 1:
            raise ValueError("initial_soc_fraction must lie in [0, 1].")


@dataclass(frozen=True)
class TariffSpec:
    """
    Settlement convention around the day-ahead market price.

    buy_price  = day_ahead_price + import_adder
    sell_price = day_ahead_price - export_deduction

    Setting both adjustments to zero gives the symmetric-price
    negative-control case used in Proposition 1.
    """

    import_adder_eur_per_mwh: float = 0.0
    export_deduction_eur_per_mwh: float = 0.0

    def __post_init__(self):
        if self.import_adder_eur_per_mwh < 0:
            raise ValueError("import_adder_eur_per_mwh must be non-negative.")

        if self.export_deduction_eur_per_mwh < 0:
            raise ValueError("export_deduction_eur_per_mwh must be non-negative.")


# ---------------------------------------------------------------------
# Frozen experiment-wide conventions
# ---------------------------------------------------------------------

TIMEZONE = "Europe/Brussels"
DECISION_ORIGIN_HOUR = 18
QUARTER_HOUR_HOURS = 0.25

BATTERY = BatterySpec()
SYMMETRIC_TARIFF = TariffSpec()


# ---------------------------------------------------------------------
# Fail-loud invariants
# ---------------------------------------------------------------------

assert TIMEZONE == "Europe/Brussels"
assert DECISION_ORIGIN_HOUR == 18
assert QUARTER_HOUR_HOURS == 0.25

assert BATTERY.energy_capacity_kwh > 0
assert 0 < BATTERY.charge_efficiency <= 1
assert 0 < BATTERY.discharge_efficiency <= 1
assert 0 <= BATTERY.initial_soc_fraction <= 1

assert SYMMETRIC_TARIFF.import_adder_eur_per_mwh == 0.0
assert SYMMETRIC_TARIFF.export_deduction_eur_per_mwh == 0.0


print("Battery specification")
print("---------------------")
print(BATTERY)

print("\nSymmetric-price negative-control tariff")
print("---------------------------------------")
print(SYMMETRIC_TARIFF)

print("\nExperiment-wide conventions")
print("---------------------------")
print(f"Decision origin : {DECISION_ORIGIN_HOUR:02d}:00 {TIMEZONE} on D-1")
print(f"Physical step   : {QUARTER_HOUR_HOURS:.2f} h")

Battery specification
---------------------
BatterySpec(energy_capacity_kwh=100.0, max_charge_kw=50.0, max_discharge_kw=50.0, charge_efficiency=0.95, discharge_efficiency=0.95, initial_soc_fraction=0.5)

Symmetric-price negative-control tariff
---------------------------------------
TariffSpec(import_adder_eur_per_mwh=0.0, export_deduction_eur_per_mwh=0.0)

Experiment-wide conventions
---------------------------
Decision origin : 18:00 Europe/Brussels on D-1
Physical step   : 0.25 h


<!-- CELL 03 — Belgian day-ahead price and information contract -->

<div style="
    background:#fff9db;
    border-left:4px solid #d6b656;
    padding:18px 22px;
    margin:12px 0 18px 0;
    border-radius:4px;
">

<h2 style="margin-top:0;">Belgian Day-Ahead Price Contract</h2>

<p>
The dispatch decision for delivery day <i>D</i> is taken at
<b>18:00 Europe/Brussels on D−1</b>.
Belgian SDAC day-ahead prices for day <i>D</i> are treated as known
at that origin.
</p>

<h3>Market-time regime</h3>

<p>
The physical battery model always operates on the Belgian civil
quarter-hour grid. The market-price representation changes within
the 2025 confirmation period.
</p>

<ul>
    <li>
        <b>Before 1 October 2025:</b>
        Belgian SDAC prices are hourly. Each observed hourly price is
        repeated over the four corresponding physical quarter-hours.
        No interpolation is performed.
    </li>

    <li>
        <b>From 1 October 2025:</b>
        native 15-minute Belgian SDAC prices are used.
    </li>
</ul>

<h3>Temporal-integrity rule</h3>

<p>
Only prices available by the 18:00 D−1 decision origin may enter an
operational dispatch calculation.
</p>

<p>
Realized load remains unknown at the decision origin and is used only
for ex-post economic evaluation.
</p>

<h3>Time-zone and DST rule</h3>

<ul>
    <li>All operational timestamps are interpreted in <b>Europe/Brussels</b>.</li>
    <li>Spring-forward civil days contain 92 quarter-hours.</li>
    <li>Ordinary civil days contain 96 quarter-hours.</li>
    <li>Fall-back civil days contain 100 quarter-hours.</li>
    <li>No day is silently normalized to 96 observations.</li>
</ul>

<h3>Data provenance</h3>

<p>
Day-ahead prices are acquired from the ENTSO-E Transparency Platform.
Raw responses will be cached locally before transformation so that
later source revisions cannot silently alter an already reported experiment.
</p>

</div>

In [2]:
# CELL 04 — ENTSO-E acquisition gate

import os
from datetime import date


ENTSOE_API_ENV_VAR = "ENTSOE_API_TOKEN"

SDAC_15MIN_FIRST_DELIVERY_DAY = date(2025, 10, 1)

PRICE_INFORMATION_AVAILABLE_AT_ORIGIN = True


# ---------------------------------------------------------------------
# Credential gate
# ---------------------------------------------------------------------

entsoe_token = os.environ.get(ENTSOE_API_ENV_VAR)

if entsoe_token:
    credential_status = "READY"
    credential_message = (
        f"{ENTSOE_API_ENV_VAR} is available in the local environment. "
        "The token value is deliberately not displayed."
    )
else:
    credential_status = "WAITING"
    credential_message = (
        f"{ENTSOE_API_ENV_VAR} is not present in the local environment. "
        "ENTSO-E acquisition remains disabled until API access is activated."
    )


# ---------------------------------------------------------------------
# Fail-loud information-contract assertions
# ---------------------------------------------------------------------

assert TIMEZONE == "Europe/Brussels"
assert DECISION_ORIGIN_HOUR == 18
assert QUARTER_HOUR_HOURS == 0.25

assert PRICE_INFORMATION_AVAILABLE_AT_ORIGIN is True
assert SDAC_15MIN_FIRST_DELIVERY_DAY == date(2025, 10, 1)


# ---------------------------------------------------------------------
# Status report
# ---------------------------------------------------------------------

print("Belgian day-ahead price contract")
print("--------------------------------")
print(f"Decision origin              : {DECISION_ORIGIN_HOUR:02d}:00 {TIMEZONE} on D-1")
print(f"Prices known at origin       : {PRICE_INFORMATION_AVAILABLE_AT_ORIGIN}")
print(f"15-minute SDAC starts        : {SDAC_15MIN_FIRST_DELIVERY_DAY}")
print("Pre-transition price mapping : hourly price repeated over 4 quarter-hours")
print("DST policy                   : preserve 92 / 96 / 100 civil quarter-hours")

print("\nENTSO-E acquisition gate")
print("------------------------")
print(f"Status : {credential_status}")
print(credential_message)

Belgian day-ahead price contract
--------------------------------
Decision origin              : 18:00 Europe/Brussels on D-1
Prices known at origin       : True
15-minute SDAC starts        : 2025-10-01
Pre-transition price mapping : hourly price repeated over 4 quarter-hours
DST policy                   : preserve 92 / 96 / 100 civil quarter-hours

ENTSO-E acquisition gate
------------------------
Status : WAITING
ENTSOE_API_TOKEN is not present in the local environment. ENTSO-E acquisition remains disabled until API access is activated.


<!-- CELL 05 — Proposition 1: forecast irrelevance under symmetric linear pricing -->

<div style="
    background:#fff9db;
    border-left:4px solid #d6b656;
    padding:18px 22px;
    margin:12px 0 18px 0;
    border-radius:4px;
">

<h2 style="margin-top:0;">
Proposition 1 — Forecast irrelevance under symmetric linear pricing
</h2>

<p>
Consider a battery dispatch problem over physical intervals
<i>t</i> = 1, ..., <i>T</i>. Let
</p>

<ul>
    <li><i>L̂<sub>t</sub></i> be forecast load,</li>
    <li><i>c<sub>t</sub></i> be battery charging power,</li>
    <li><i>d<sub>t</sub></i> be battery discharging power,</li>
    <li><i>g<sub>t</sub></i> be signed grid exchange, positive for import,</li>
    <li><i>p<sub>t</sub></i> be the single linear settlement price.</li>
</ul>

<p>
Grid exchange satisfies
</p>

<p style="text-align:center; font-size:1.08em;">
<b>
g<sub>t</sub> =
L̂<sub>t</sub> + c<sub>t</sub> − d<sub>t</sub>.
</b>
</p>

<p>
Suppose that:
</p>

<ol>
    <li>grid exchange is unconstrained in sign;</li>
    <li>imports and exports are settled at the same linear price
        <i>p<sub>t</sub></i>;</li>
    <li>battery constraints are independent of load;</li>
    <li>the terminal state-of-charge condition is independent of load.</li>
</ol>

<p>
Then the optimal battery dispatch is <b>independent of the load forecast</b>.
</p>

<h3>Proof</h3>

<p>
The forecast-based energy cost is
</p>

<p style="text-align:center;">
<b>
Σ<sub>t</sub>
p<sub>t</sub> g<sub>t</sub> Δt.
</b>
</p>

<p>
Substituting the grid-balance equation gives
</p>

<p style="text-align:center;">
<b>
Σ<sub>t</sub>
p<sub>t</sub>L̂<sub>t</sub>Δt
&nbsp;+&nbsp;
Σ<sub>t</sub>
p<sub>t</sub>(c<sub>t</sub>−d<sub>t</sub>)Δt.
</b>
</p>

<p>
The first term depends on the load forecast but contains no battery
decision variable. It is therefore an additive constant in the
optimization problem.
</p>

<p>
Because signed grid exchange is unconstrained, the balance equation
merely determines <i>g<sub>t</sub></i> for any battery-feasible
(<i>c</i>, <i>d</i>) trajectory. It imposes no additional
load-dependent restriction on battery dispatch.
</p>

<p>
The minimizing battery trajectory therefore depends on prices and
battery constraints, but not on the load forecast.
<b>□</b>
</p>

<hr style="border:none; border-top:1px solid #e2d690; margin:20px 0;">

<h3>Interpretation</h3>

<p>
This result gives the project an important negative control.
A load forecast does not automatically possess economic decision value
merely because it appears in the power-balance equation.
</p>

<p>
Forecast value arises only when the separation above is broken.
Relevant mechanisms include:
</p>

<ul>
    <li>different import and export prices;</li>
    <li>grid import or export capacity constraints;</li>
    <li>load-dependent operating constraints;</li>
    <li>nonlinear tariffs such as demand or peak charges.</li>
</ul>

<p>
Version 1 deliberately introduces the first mechanism through
</p>

<p style="text-align:center;">
<b>
p<sup>buy</sup><sub>t</sub>
= p<sup>DA</sup><sub>t</sub> + τ<sub>b</sub>,
&nbsp;&nbsp;&nbsp;
p<sup>sell</sup><sub>t</sub>
= p<sup>DA</sup><sub>t</sub> − τ<sub>s</sub>,
</b>
</p>

<p>
with τ<sub>b</sub> ≥ 0 and τ<sub>s</sub> ≥ 0.
The symmetric case τ<sub>b</sub> = τ<sub>s</sub> = 0 is retained
as the negative control.
</p>

<h3>Numerical falsification plan</h3>

<p>
<b>NC-1 — Symmetric-price negative control.</b><br>
For the same prices and battery specification, replace one load
forecast by a materially different forecast while setting
τ<sub>b</sub> = τ<sub>s</sub> = 0.
After deterministic tie-breaking, the resulting battery dispatches
must agree within numerical tolerance.
</p>

<p>
<b>PC-1 — Asymmetric-price positive control.</b><br>
Construct a case with non-zero import/export spread in which alternative
load forecasts move net grid exchange across the import/export kink.
At least one such case must produce a different optimal dispatch.
</p>

<p>
Failure of NC-1 indicates an implementation, formulation, or
tie-breaking problem. Failure of PC-1 indicates that the intended
economic mechanism has not actually been activated.
</p>

<h3>Consequence for forecast evaluation</h3>

<p>
Conventional forecast-error measures such as MAE treat errors at
different times symmetrically. Battery economics need not.
The economic effect of a forecast error depends on prices, state of
charge, active constraints, and whether the error changes the relevant
import/export regime.
</p>

<p style="
    text-align:center;
    font-size:1.08em;
    margin-bottom:0;
">
<b>
Forecast accuracy and forecast decision value are therefore
distinct quantities.
</b>
</p>

</div>

<!-- CELL 06 — Mathematical dispatch specification -->

<div style="
    background:#fff9db;
    border-left:4px solid #d6b656;
    padding:18px 22px;
    margin:12px 0 18px 0;
    border-radius:4px;
">

<h2 style="margin-top:0;">Minimal Battery-Dispatch Model</h2>

<p>
For each physical quarter-hour
<i>t</i> = 1, ..., <i>T</i>,
the operational optimization chooses battery charging and discharging
subject to power, energy and terminal-state constraints.
</p>

<h3>Decision variables</h3>

<ul>
    <li>
        <i>c<sub>t</sub></i> ≥ 0:
        battery charging power in kW;
    </li>
    <li>
        <i>d<sub>t</sub></i> ≥ 0:
        battery discharging power in kW;
    </li>
    <li>
        <i>s<sub>t</sub></i>:
        battery state of charge in kWh;
    </li>
    <li>
        <i>g<sup>+</sup><sub>t</sub></i> ≥ 0:
        grid import power in kW;
    </li>
    <li>
        <i>g<sup>−</sup><sub>t</sub></i> ≥ 0:
        grid export power in kW.
    </li>
</ul>

<h3>Power balance</h3>

<p style="text-align:center; font-size:1.08em;">
<b>
g<sup>+</sup><sub>t</sub>
−
g<sup>−</sup><sub>t</sub>
+
d<sub>t</sub>
−
c<sub>t</sub>
=
L̂<sub>t</sub>.
</b>
</p>

<p>
The optimization uses the load forecast
<i>L̂<sub>t</sub></i>, not realized future load.
</p>

<h3>State-of-charge dynamics</h3>

<p style="text-align:center; font-size:1.08em;">
<b>
s<sub>t+1</sub>
=
s<sub>t</sub>
+
η<sub>c</sub> c<sub>t</sub> Δt
−
d<sub>t</sub> Δt / η<sub>d</sub>.
</b>
</p>

<p>
Charging and discharging efficiencies are represented explicitly.
The power variables are defined on the grid/battery interface according
to this convention and will be used consistently throughout the project.
</p>

<h3>Battery constraints</h3>

<p style="text-align:center;">
<b>
0 ≤ c<sub>t</sub> ≤ C̄,
&nbsp;&nbsp;&nbsp;
0 ≤ d<sub>t</sub> ≤ D̄,
</b>
</p>

<p style="text-align:center;">
<b>
0 ≤ s<sub>t</sub> ≤ Ē.
</b>
</p>

<p>
The daily experiment fixes the initial state of charge and imposes
a terminal-state condition
</p>

<p style="text-align:center;">
<b>
s<sub>T+1</sub> = s<sub>1</sub>.
</b>
</p>

<p>
This prevents the optimizer from manufacturing apparent daily savings
by ending the day with a systematically depleted battery.
</p>

<h3>Economic objective</h3>

<p>
Let
</p>

<p style="text-align:center;">
<b>
p<sup>buy</sup><sub>t</sub>
=
p<sup>DA</sup><sub>t</sub> + τ<sub>b</sub>,
</b>
</p>

<p style="text-align:center;">
<b>
p<sup>sell</sup><sub>t</sub>
=
p<sup>DA</sup><sub>t</sub> − τ<sub>s</sub>.
</b>
</p>

<p>
The dispatch minimizes forecast-based day-ahead settlement cost:
</p>

<p style="text-align:center; font-size:1.08em;">
<b>
min
&nbsp;
Σ<sub>t</sub>
[
p<sup>buy</sup><sub>t</sub> g<sup>+</sup><sub>t</sub>
−
p<sup>sell</sup><sub>t</sub> g<sup>−</sup><sub>t</sub>
]
Δt / 1000.
</b>
</p>

<p>
The factor 1/1000 converts kWh-valued interval energy into MWh,
because prices are expressed in €/MWh.
</p>

<h3>Open-loop execution</h3>

<p>
Once the D−1 optimization has produced
(<i>c<sub>t</sub></i>, <i>d<sub>t</sub></i>),
that battery schedule is frozen for delivery day <i>D</i>.
</p>

<p>
Realized load
<i>L<sub>t</sub></i>
changes only realized grid exchange:
</p>

<p style="text-align:center; font-size:1.08em;">
<b>
g<sup>real</sup><sub>t</sub>
=
L<sub>t</sub>
+
c<sub>t</sub>
−
d<sub>t</sub>.
</b>
</p>

<p>
The battery schedule is not re-optimized using information that
arrives after the decision origin.
</p>

<h3>Realized economic value</h3>

<p>
The realized cost of a frozen dispatch is calculated from realized
grid exchange using the same import/export settlement rule.
Economic forecast value is therefore measured through the realized
consequence of the decision produced by each forecast.
</p>

<p style="
    text-align:center;
    font-size:1.08em;
    margin-bottom:0;
">
<b>
forecast → planned dispatch → realized grid exchange → realized cost
</b>
</p>

</div>

<!-- CELL 07 — Implementation contract and edge cases -->

<div style="
    background:#fff9db;
    border-left:4px solid #d6b656;
    padding:18px 22px;
    margin:12px 0 18px 0;
    border-radius:4px;
">

<h2 style="margin-top:0;">
Implementation Contract and Edge Cases
</h2>

<h3>Units</h3>

<ul>
    <li>Load, charge, discharge and grid power: <b>kW</b>.</li>
    <li>Battery state of charge: <b>kWh</b>.</li>
    <li>Day-ahead and settlement prices: <b>€/MWh</b>.</li>
    <li>Physical interval duration: <b>0.25 h</b>.</li>
    <li>Optimization objective and realized economic cost: <b>€</b>.</li>
</ul>

<h3>Sign and efficiency conventions</h3>

<p>
Charging power <i>c<sub>t</sub></i> and discharging power
<i>d<sub>t</sub></i> are measured at the external battery power interface.
Positive charging increases grid demand; positive discharging reduces it.
</p>

<p>
Battery state evolves according to
</p>

<p style="text-align:center; font-size:1.08em;">
<b>
s<sub>t+1</sub>
=
s<sub>t</sub>
+
η<sub>c</sub> c<sub>t</sub> Δt
−
d<sub>t</sub> Δt / η<sub>d</sub>.
</b>
</p>

<p>
This convention is fixed throughout the project. Efficiency losses are
therefore represented inside the battery state equation rather than by
redefining external charge or discharge power.
</p>

<h3>Import and export</h3>

<p>
Grid import and export are represented by separate non-negative variables.
When
</p>

<p style="text-align:center;">
<b>
p<sup>buy</sup><sub>t</sub>
≥
p<sup>sell</sup><sub>t</sub>,
</b>
</p>

<p>
simultaneous import and export are economically unnecessary. The
implementation nevertheless checks returned solutions for material
simultaneous import/export rather than relying only on that argument.
</p>

<h3>Lexicographic identification of the dispatch</h3>

<p>
A linear program can have multiple economically equivalent optimal
solutions. This matters because two solver-selected representatives of
the same optimal set must not be interpreted as different economic
responses to different forecasts.
</p>

<p>
The dispatch is therefore identified using a three-stage lexicographic
hierarchy:
</p>

<ol>
    <li>
        <b>Primary:</b> minimize economic settlement cost.
    </li>
    <li>
        <b>Secondary:</b> among solutions within the fixed numerical
        tolerance of the primary optimum, minimize total battery
        throughput
        Σ(c<sub>t</sub> + d<sub>t</sub>)Δt.
    </li>
    <li>
        <b>Tertiary:</b> among the remaining solutions, minimize a fixed
        temporal timing score, thereby selecting a deterministic
        representative of otherwise equivalent schedules.
    </li>
</ol>

<p>
The secondary and tertiary objectives do not replace or perturb the
economic objective. They are solved as subsequent optimization problems
subject to preservation of the preceding optimum within explicitly
defined numerical tolerances.
</p>

<p>
No arbitrary epsilon penalty is added to the primary economic objective.
</p>

<h3>Negative prices and simultaneous battery cycling</h3>

<p>
The continuous LP does not explicitly prohibit simultaneous charging and
discharging.
</p>

<p>
Under negative settlement prices, round-trip losses can make such cycling
economically optimal: the model can deliberately dissipate energy in
order to consume additional negatively priced electricity.
</p>

<p>
This behaviour has been derived analytically and reproduced in a
constructed numerical stress test. It is therefore treated as a known
property of the present formulation rather than as a solver defect.
</p>

<p>
Every empirical dispatch will be checked for material simultaneous
charging and discharging. The Belgian price data will determine whether
an explicit mutually exclusive operating-mode formulation is required.
</p>

<h3>Independent solution checks</h3>

<p>
Every dispatch result is subject to independent reconstruction or
validation of:
</p>

<ul>
    <li>power balance;</li>
    <li>state-of-charge recursion;</li>
    <li>charge, discharge and state-of-charge bounds;</li>
    <li>terminal state of charge;</li>
    <li>economic objective reconstruction;</li>
    <li>simultaneous import and export;</li>
    <li>simultaneous charging and discharging.</li>
</ul>

<h3>Temporal-integrity check</h3>

<p>
Realized post-origin load is an evaluation input, not an optimization
input. A later leakage test will perturb realized future load by an
extreme amount and verify that the planned battery dispatch remains
unchanged.
</p>

<p style="
    text-align:center;
    font-size:1.08em;
    margin-bottom:0;
">
<b>
The implementation should make an invalid economic experiment difficult
to perform accidentally.
</b>
</p>

</div>

In [3]:
# CELL 08 — First verified dispatch experiment

import numpy as np

from battery_dispatch_forecast_value.dispatch import (
    DispatchInputs,
    solve_dispatch,
)


def notebook_dispatch_inputs(
    load_kw,
    price_eur_per_mwh,
    *,
    import_adder=0.0,
    export_deduction=0.0,
):
    """
    Construct a small dispatch problem using the experiment-wide
    battery specification defined in CELL 02.
    """

    return DispatchInputs(
        forecast_load_kw=np.asarray(load_kw, dtype=float),
        day_ahead_price_eur_per_mwh=np.asarray(
            price_eur_per_mwh,
            dtype=float,
        ),
        energy_capacity_kwh=BATTERY.energy_capacity_kwh,
        max_charge_kw=BATTERY.max_charge_kw,
        max_discharge_kw=BATTERY.max_discharge_kw,
        charge_efficiency=BATTERY.charge_efficiency,
        discharge_efficiency=BATTERY.discharge_efficiency,
        initial_soc_kwh=(
            BATTERY.initial_soc_fraction
            * BATTERY.energy_capacity_kwh
        ),
        terminal_soc_kwh=(
            BATTERY.initial_soc_fraction
            * BATTERY.energy_capacity_kwh
        ),
        interval_hours=QUARTER_HOUR_HOURS,
        import_adder_eur_per_mwh=import_adder,
        export_deduction_eur_per_mwh=export_deduction,
    )


# ---------------------------------------------------------------------
# Shared stylised price path and two deliberately different forecasts
# ---------------------------------------------------------------------

price = np.array(
    [20.0, 20.0, 40.0, 80.0, 160.0, 80.0, 40.0, 20.0]
)

forecast_a = np.array(
    [20.0, 30.0, 40.0, 50.0, 60.0, 50.0, 30.0, 20.0]
)

forecast_b = np.array(
    [150.0, 5.0, 100.0, 10.0, 200.0, 5.0, 120.0, 1.0]
)


# ---------------------------------------------------------------------
# NC-1 — symmetric-price negative control
# ---------------------------------------------------------------------

nc_a = solve_dispatch(
    notebook_dispatch_inputs(forecast_a, price)
)

nc_b = solve_dispatch(
    notebook_dispatch_inputs(forecast_b, price)
)

nc_charge_difference = np.max(
    np.abs(nc_a.charge_kw - nc_b.charge_kw)
)

nc_discharge_difference = np.max(
    np.abs(nc_a.discharge_kw - nc_b.discharge_kw)
)

nc_soc_difference = np.max(
    np.abs(nc_a.soc_kwh - nc_b.soc_kwh)
)


# ---------------------------------------------------------------------
# PC-1 — asymmetric-price positive control
# ---------------------------------------------------------------------

pc_price = np.array([20.0, 100.0])

pc_low_forecast = np.array([0.0, 0.0])
pc_high_forecast = np.array([100.0, 100.0])

pc_low = solve_dispatch(
    notebook_dispatch_inputs(
        pc_low_forecast,
        pc_price,
        import_adder=50.0,
        export_deduction=50.0,
    )
)

pc_high = solve_dispatch(
    notebook_dispatch_inputs(
        pc_high_forecast,
        pc_price,
        import_adder=50.0,
        export_deduction=50.0,
    )
)

pc_charge_difference = np.max(
    np.abs(pc_low.charge_kw - pc_high.charge_kw)
)

pc_discharge_difference = np.max(
    np.abs(pc_low.discharge_kw - pc_high.discharge_kw)
)

pc_dispatch_difference = max(
    pc_charge_difference,
    pc_discharge_difference,
)


# ---------------------------------------------------------------------
# Fail loudly if either theoretical control is violated
# ---------------------------------------------------------------------

assert nc_charge_difference < 1e-6
assert nc_discharge_difference < 1e-6
assert nc_soc_difference < 1e-6

assert pc_dispatch_difference > 1e-6


# ---------------------------------------------------------------------
# Compact research record
# ---------------------------------------------------------------------

print("NC-1 — symmetric-price negative control")
print("-----------------------------------------")
print(
    f"Maximum charge difference    : "
    f"{nc_charge_difference:.3e} kW"
)
print(
    f"Maximum discharge difference : "
    f"{nc_discharge_difference:.3e} kW"
)
print(
    f"Maximum SoC difference       : "
    f"{nc_soc_difference:.3e} kWh"
)
print("Result                       : PASS")

print("\nPC-1 — asymmetric-price positive control")
print("-----------------------------------------")
print(
    f"Maximum charge difference    : "
    f"{pc_charge_difference:.6f} kW"
)
print(
    f"Maximum discharge difference : "
    f"{pc_discharge_difference:.6f} kW"
)
print(
    f"Maximum dispatch difference  : "
    f"{pc_dispatch_difference:.6f} kW"
)
print("Result                       : PASS")

NC-1 — symmetric-price negative control
-----------------------------------------
Maximum charge difference    : 3.197e-14 kW
Maximum discharge difference : 2.842e-14 kW
Maximum SoC difference       : 1.421e-14 kWh
Result                       : PASS

PC-1 — asymmetric-price positive control
-----------------------------------------
Maximum charge difference    : 49.999999 kW
Maximum discharge difference : 45.124999 kW
Maximum dispatch difference  : 49.999999 kW
Result                       : PASS


<!-- CELL 09 — Research checkpoint: forecast irrelevance verified -->

<div style="
    background:#fff9db;
    border-left:4px solid #d6b656;
    padding:18px 22px;
    margin:12px 0 18px 0;
    border-radius:4px;
">

<h2 style="margin-top:0;">
Research Checkpoint — Forecast Irrelevance Verified
</h2>

<p>
The analytical result in Proposition 1 has now been reproduced by the
implemented linear-programming model.
</p>

<h3>Negative control NC-1</h3>

<p>
Two deliberately and materially different load forecasts were optimized
against the same battery specification and the same price path under
symmetric linear settlement.
</p>

<p>
After complete lexicographic tie-breaking, the maximum differences between
the resulting battery schedules were of order 10<sup>−14</sup>:
</p>

<ul>
    <li>maximum charging-power difference: approximately 3.2 × 10<sup>−14</sup> kW;</li>
    <li>maximum discharging-power difference: approximately 2.8 × 10<sup>−14</sup> kW;</li>
    <li>maximum state-of-charge difference: approximately 1.4 × 10<sup>−14</sup> kWh.</li>
</ul>

<p>
These differences are numerical floating-point noise. Operationally,
the dispatches are identical.
</p>

<h3>Positive control PC-1</h3>

<p>
The same optimization framework was also tested under asymmetric
import/export settlement. A constructed pair of forecasts crossing
the import/export kink produced different optimal battery schedules.
</p>

<p>
The implementation therefore reproduces both sides of the intended
economic mechanism:
</p>

<p style="text-align:center; font-size:1.08em;">
<b>
symmetric settlement → forecast-irrelevant dispatch
</b>
</p>

<p style="text-align:center; font-size:1.08em;">
<b>
asymmetric settlement → forecast-dependent dispatch can occur
</b>
</p>

<h3>Solver degeneracy is not economic dependence</h3>

<p>
The first implementation minimized economic cost and then total battery
throughput. That was insufficient to identify a unique dispatch when
several time intervals were economically equivalent.
</p>

<p>
Two forecasts consequently produced schedules differing by approximately
1.1 × 10<sup>−6</sup> kW even though they represented the same economic
solution.
</p>

<p>
The solver was therefore strengthened to use a three-stage lexicographic
hierarchy:
</p>

<ol>
    <li><b>minimize economic cost;</b></li>
    <li><b>among economic optima, minimize battery throughput;</b></li>
    <li><b>among the remaining optima, apply a fixed temporal tie-break.</b></li>
</ol>

<p>
After this change, NC-1 differences collapsed to machine-level numerical
noise.
</p>

<p>
This distinction matters for the later forecast comparison:
<b>different solver-selected representatives of the same optimal set must
not be mistaken for economic sensitivity to the forecast.</b>
</p>

<h3>Current validation status</h3>

<p>
The dispatch implementation currently passes <b>10/10 automated tests</b>,
covering power balance, state-of-charge dynamics, terminal state,
objective reconstruction, arbitrage behaviour, input validation,
NC-1 and PC-1.
</p>

<p style="
    text-align:center;
    font-size:1.08em;
    margin-bottom:0;
">
<b>
The model can produce both zero and non-zero forecast decision value
for mathematically understood reasons.
</b>
</p>

</div>

<!-- CELL 10 — Negative prices and simultaneous battery cycling -->

<div style="
    background:#fff9db;
    border-left:4px solid #d6b656;
    padding:18px 22px;
    margin:12px 0 18px 0;
    border-radius:4px;
">

<h2 style="margin-top:0;">
Negative Prices and Simultaneous Battery Cycling
</h2>

<p>
The linear-programming formulation does not explicitly prohibit
simultaneous charging and discharging. Under ordinary positive prices
this is normally dominated by avoiding unnecessary round-trip losses.
Negative prices require separate analysis.
</p>

<h3>A state-of-charge-neutral simultaneous cycle</h3>

<p>
Consider one interval in which charging and discharging are both positive,
while the battery state of charge is left unchanged by that simultaneous
activity.
</p>

<p>
For an additional charging quantity
<i>δc</i> &gt; 0, state-of-charge neutrality requires
</p>

<p style="text-align:center; font-size:1.08em;">
<b>
η<sub>c</sub> δc
−
δd / η<sub>d</sub>
=
0,
</b>
</p>

<p>
and therefore
</p>

<p style="text-align:center; font-size:1.08em;">
<b>
δd
=
η<sub>c</sub> η<sub>d</sub> δc.
</b>
</p>

<p>
Because η<sub>c</sub>η<sub>d</sub> &lt; 1, the simultaneous cycle consumes
more external electrical energy than it returns.
</p>

<p>
Its additional net grid import is
</p>

<p style="text-align:center; font-size:1.08em;">
<b>
δg
=
δc − δd
=
(1 − η<sub>c</sub>η<sub>d</sub>) δc
&gt; 0.
</b>
</p>

<h3>Economic consequence while importing</h3>

<p>
Suppose the site remains on the import side of the settlement kink.
The incremental cost of this state-of-charge-neutral cycle is
</p>

<p style="text-align:center; font-size:1.08em;">
<b>
δC
=
p<sup>buy</sup><sub>t</sub>
(1 − η<sub>c</sub>η<sub>d</sub>)
δc Δt / 1000.
</b>
</p>

<p>
Since
1 − η<sub>c</sub>η<sub>d</sub> &gt; 0,
the sign of the incremental cost is determined by the buy price.
</p>

<ul>
    <li>
        If <i>p<sup>buy</sup><sub>t</sub></i> &gt; 0,
        simultaneous cycling increases cost and is dominated.
    </li>
    <li>
        If <i>p<sup>buy</sup><sub>t</sub></i> = 0,
        the primary economic objective is indifferent.
    </li>
    <li>
        If <i>p<sup>buy</sup><sub>t</sub></i> &lt; 0,
        simultaneous cycling can <b>reduce</b> cost by deliberately
        dissipating energy.
    </li>
</ul>

<h3>Economic consequence while exporting</h3>

<p>
On the export side, increasing net grid import means reducing the
magnitude of export. The same local argument shows that the relevant
marginal settlement price is the sell price.
</p>

<p>
If
<i>p<sup>sell</sup><sub>t</sub></i> &lt; 0,
reducing export of negatively priced energy can also be economically
valuable. Behaviour at the import/export kink must therefore be evaluated
using the complete piecewise-linear settlement objective rather than a
single global price rule.
</p>

<h3>Implication for the LP formulation</h3>

<p>
Simultaneous charging and discharging is not merely a numerical pathology.
Under negative settlement prices it can be an economically optimal way
for the mathematical model to consume energy while keeping state of
charge approximately unchanged.
</p>

<p>
Whether that behaviour is physically admissible depends on the intended
battery-controller model. A real battery energy-management system would
normally not be treated as an unrestricted device for simultaneous
charge/discharge energy destruction.
</p>

<p>
Consequently, version 1 adopts the following rule:
</p>

<p style="
    text-align:center;
    font-size:1.08em;
">
<b>
The LP remains the preferred formulation only where its optimal solution
contains no economically material simultaneous charging and discharging.
</b>
</p>

<p>
Every empirical dispatch will therefore be checked for
</p>

<p style="text-align:center;">
<b>
min(c<sub>t</sub>, d<sub>t</sub>) &gt; tolerance.
</b>
</p>

<p>
If Belgian price observations cause material simultaneous cycling,
the case will be reported rather than silently removed. We will then
decide whether the operational model requires an explicit mutually
exclusive charge/discharge formulation.
</p>

<h3>Why no binary variable yet?</h3>

<p>
Adding a binary operating-mode variable would convert the problem from
a linear program into a mixed-integer linear program. That may eventually
be justified, but introducing it before observing the pathology would
increase computational and methodological complexity unnecessarily.
</p>

<p>
The preferred sequence is therefore:
</p>

<ol>
    <li>derive the failure condition analytically;</li>
    <li>construct a numerical case that deliberately activates it;</li>
    <li>verify that the implementation detects it;</li>
    <li>inspect its relevance in the Belgian empirical price domain;</li>
    <li>strengthen the formulation only if required.</li>
</ol>

<p style="
    text-align:center;
    font-size:1.08em;
    margin-bottom:0;
">
<b>
Negative prices are a model test, not a data-cleaning problem.
</b>
</p>

</div>

In [4]:
# CELL 11 — Negative-price cycling stress test

import numpy as np

from battery_dispatch_forecast_value.dispatch import (
    DispatchInputs,
    solve_dispatch,
)


# ---------------------------------------------------------------------
# Construct a deliberately hostile case
# ---------------------------------------------------------------------
#
# Four quarter-hours, all at a strongly negative price.
#
# The battery starts and must finish at 50 kWh.
# Load is high enough to keep the site on the import side, so the
# relevant marginal settlement price is the negative buy price.
#
# Under the current LP, simultaneous charging and discharging can
# become economically attractive because round-trip losses allow
# additional paid consumption without increasing terminal SoC.
# ---------------------------------------------------------------------

T_NEG = 4

negative_price = np.full(T_NEG, -200.0)
negative_load = np.full(T_NEG, 200.0)

negative_case = DispatchInputs(
    forecast_load_kw=negative_load,
    day_ahead_price_eur_per_mwh=negative_price,
    energy_capacity_kwh=BATTERY.energy_capacity_kwh,
    max_charge_kw=BATTERY.max_charge_kw,
    max_discharge_kw=BATTERY.max_discharge_kw,
    charge_efficiency=BATTERY.charge_efficiency,
    discharge_efficiency=BATTERY.discharge_efficiency,
    initial_soc_kwh=(
        BATTERY.initial_soc_fraction
        * BATTERY.energy_capacity_kwh
    ),
    terminal_soc_kwh=(
        BATTERY.initial_soc_fraction
        * BATTERY.energy_capacity_kwh
    ),
    interval_hours=QUARTER_HOUR_HOURS,
    import_adder_eur_per_mwh=0.0,
    export_deduction_eur_per_mwh=0.0,
)

negative_result = solve_dispatch(negative_case)


# ---------------------------------------------------------------------
# Diagnose simultaneous charging and discharging
# ---------------------------------------------------------------------

simultaneous_cycle_kw = np.minimum(
    negative_result.charge_kw,
    negative_result.discharge_kw,
)

max_simultaneous_cycle_kw = float(
    np.max(simultaneous_cycle_kw)
)

cycling_intervals = np.flatnonzero(
    simultaneous_cycle_kw > 1e-6
)


# ---------------------------------------------------------------------
# Independently reconstruct state-of-charge changes
# ---------------------------------------------------------------------

soc_change_kwh = np.diff(
    negative_result.soc_kwh
)

reconstructed_soc_change_kwh = (
    BATTERY.charge_efficiency
    * negative_result.charge_kw
    * QUARTER_HOUR_HOURS
    -
    negative_result.discharge_kw
    * QUARTER_HOUR_HOURS
    / BATTERY.discharge_efficiency
)

assert np.allclose(
    soc_change_kwh,
    reconstructed_soc_change_kwh,
    atol=1e-7,
)

assert np.isclose(
    negative_result.soc_kwh[-1],
    negative_result.soc_kwh[0],
    atol=1e-7,
)


# ---------------------------------------------------------------------
# Research record
# ---------------------------------------------------------------------

print("Negative-price cycling stress test")
print("----------------------------------")
print(
    f"Day-ahead price        : "
    f"{negative_price[0]:.2f} EUR/MWh"
)
print(
    f"Forecast load          : "
    f"{negative_load[0]:.2f} kW"
)
print(
    f"Round-trip efficiency  : "
    f"{BATTERY.charge_efficiency * BATTERY.discharge_efficiency:.4f}"
)
print(
    f"Cycle loss fraction    : "
    f"{1.0 - BATTERY.charge_efficiency * BATTERY.discharge_efficiency:.4f}"
)

print("\nDispatch")
print("--------")
print(
    "Charge kW    :",
    np.round(negative_result.charge_kw, 6),
)
print(
    "Discharge kW :",
    np.round(negative_result.discharge_kw, 6),
)
print(
    "Import kW    :",
    np.round(negative_result.import_kw, 6),
)
print(
    "Export kW    :",
    np.round(negative_result.export_kw, 6),
)
print(
    "SoC kWh      :",
    np.round(negative_result.soc_kwh, 6),
)

print("\nSimultaneous cycling diagnostic")
print("--------------------------------")
print(
    f"Maximum simultaneous charge/discharge : "
    f"{max_simultaneous_cycle_kw:.6f} kW"
)
print(
    f"Intervals above 1e-6 kW                : "
    f"{cycling_intervals.tolist()}"
)

print("\nEconomic result")
print("---------------")
print(
    f"Primary optimum : "
    f"{negative_result.primary_objective_eur:.8f} EUR"
)
print(
    f"Final objective : "
    f"{negative_result.objective_eur:.8f} EUR"
)

if max_simultaneous_cycle_kw > 1e-6:
    print(
        "\nRESULT: PATHOLOGY REPRODUCED — "
        "the LP uses simultaneous charging and discharging."
    )
else:
    print(
        "\nRESULT: pathology not reproduced in this constructed case."
    )

Negative-price cycling stress test
----------------------------------
Day-ahead price        : -200.00 EUR/MWh
Forecast load          : 200.00 kW
Round-trip efficiency  : 0.9025
Cycle loss fraction    : 0.0975

Dispatch
--------
Charge kW    : [50.       50.       50.       49.999998]
Discharge kW : [50.       50.       50.       30.499998]
Import kW    : [200.  200.  200.  219.5]
Export kW    : [0. 0. 0. 0.]
SoC kWh      : [50.       48.717105 47.434211 46.151316 50.      ]

Simultaneous cycling diagnostic
--------------------------------
Maximum simultaneous charge/discharge : 50.000000 kW
Intervals above 1e-6 kW                : [0, 1, 2, 3]

Economic result
---------------
Primary optimum : -40.97500000 EUR
Final objective : -40.97499999 EUR

RESULT: PATHOLOGY REPRODUCED — the LP uses simultaneous charging and discharging.


<!-- CELL 12 — Negative-price pathology checkpoint -->

<div style="
    background:#fff9db;
    border-left:4px solid #d6b656;
    padding:18px 22px;
    margin:12px 0 18px 0;
    border-radius:4px;
">

<h2 style="margin-top:0;">
Research Checkpoint — Negative-Price Cycling Pathology Confirmed
</h2>

<p>
A deliberately constructed negative-price case was used to test whether
the continuous linear-programming formulation can exploit round-trip
battery losses.
</p>

<h3>Stress-test configuration</h3>

<ul>
    <li>Day-ahead price: <b>−200 €/MWh</b> in every interval.</li>
    <li>Forecast load: <b>200 kW</b> in every interval.</li>
    <li>Charge efficiency: <b>0.95</b>.</li>
    <li>Discharge efficiency: <b>0.95</b>.</li>
    <li>Round-trip efficiency: <b>0.9025</b>.</li>
    <li>Round-trip loss fraction: <b>0.0975</b>.</li>
    <li>Initial and terminal state of charge: <b>50 kWh</b>.</li>
</ul>

<h3>Observed behaviour</h3>

<p>
The LP selected simultaneous charging and discharging in all four
quarter-hours.
</p>

<p>
The maximum simultaneous charge/discharge level was
<b>50 kW</b>, equal to the battery power limit.
</p>

<p>
The optimizer therefore used the battery as an energy-dissipation
mechanism: it imported additional negatively priced electricity,
destroyed part of that energy through round-trip losses, and still
satisfied the terminal state-of-charge condition.
</p>

<p>
This behaviour is mathematically consistent with the objective.
It is not a numerical solver error.
</p>

<h3>Interpretation</h3>

<p>
For a state-of-charge-neutral simultaneous cycle,
</p>

<p style="text-align:center; font-size:1.08em;">
<b>
δg
=
(1 − η<sub>c</sub>η<sub>d</sub>) δc
&gt; 0.
</b>
</p>

<p>
When the relevant settlement price is negative, increasing net import
can reduce total cost. The optimizer therefore has a genuine economic
incentive to create losses.
</p>

<h3>Consequence for the production model</h3>

<p>
The continuous LP cannot automatically be assumed to represent
physically admissible battery operation over the complete Belgian
price domain.
</p>

<p>
Before deciding whether to introduce an explicit mutually exclusive
charge/discharge formulation, the empirical relevance of this pathology
must be measured.
</p>

<h3>Empirical gate</h3>

<p>
Once Belgian 2025 day-ahead prices are available, the following quantities
will be measured over the confirmation period:
</p>

<ul>
    <li>number and proportion of intervals with negative day-ahead price;</li>
    <li>number and proportion with negative effective buy price;</li>
    <li>number and proportion with negative effective sell price;</li>
    <li>number of delivery days containing at least one such interval;</li>
    <li>magnitude and duration of negative-price episodes;</li>
    <li>
        frequency with which the unconstrained LP actually returns
        material simultaneous charging and discharging.
    </li>
</ul>

<p>
This distinction is important:
<b>negative prices create the possibility of the pathology, but do not
by themselves prove that the realized optimization will use it.</b>
Battery state, load, settlement spread and neighbouring prices also matter.
</p>

<h3>Decision rule</h3>

<p>
The formulation will be strengthened only after this empirical gate:
</p>

<ol>
    <li>
        If simultaneous cycling is absent or economically immaterial in
        the empirical domain, retain the LP as the primary formulation
        and report the diagnostic.
    </li>
    <li>
        If simultaneous cycling occurs materially, introduce an explicit
        mutually exclusive charge/discharge formulation and quantify the
        effect of that change.
    </li>
</ol>

<p style="
    text-align:center;
    font-size:1.08em;
    margin-bottom:0;
">
<b>
Model complexity will be triggered by observed economic relevance,
not introduced pre-emptively.
</b>
</p>

</div>